# UC1 defect inspection playground

This notebook demonstrates the simulation framework in a concrete inspection use case:
illumination → surface → scattering → optics → detector → analysis.

Use this notebook as a play space: change the parameters in the next cell and rerun the later cells to see how the result changes.

## How to use this notebook

- Edit the parameters in the next cell to change the defect type, illumination geometry, and defect threshold.
- Run the cells in order.
- Compare how the same framework behaves for different settings.

In [ ]:
# Editable parameters: change these values and rerun the notebook.
shape = (64, 64)
defect_type = "dent"  # options: "dent", "pit", "crack", "stain"
illumination = "darkfield"  # options: "brightfield", "darkfield"
threshold = 0.08
exposure_time = 1e-5

print("Configuration:")
print(f"  defect_type={defect_type}")
print(f"  illumination={illumination}")
print(f"  threshold={threshold}")

In [ ]:
# Import the framework components that form the pipeline.
import numpy as np

from optical_metrology.analysis import DefectAnalyzer
from optical_metrology.detector import CMOSDetector
from optical_metrology.illumination import bright_field, dark_field
from optical_metrology.optics import GaussianPSF, OpticalPropagator, OpticalSystem
from optical_metrology.pipeline import SimulationPipeline
from optical_metrology.scattering import LambertianScattering
from optical_metrology.surface import CrackSurface, DentSurface, Material, PitSurface, StainSurface

In [ ]:
# Build the surface from the selected defect type.

surface_map = {
    "dent": DentSurface,
    "pit": PitSurface,
    "crack": CrackSurface,
    "stain": StainSurface,
}

surface_cls = surface_map[defect_type]

# The surface generator creates a height map and derived geometry.
# Increasing the defect depth or radius makes it easier to detect.
surface = surface_cls(
    shape=shape,
    material=Material("silicon"),
    depth=0.5 if defect_type in {"dent", "pit"} else 0.25,
    radius=4.0 if defect_type in {"dent", "pit"} else 6.0,
)

print(f"Surface type: {surface_cls.__name__}")
print(f"Surface roughness: {getattr(surface, 'roughness', float('nan')):.4f}")

In [ ]:
# Build the illumination source for the selected geometry.
# This is where the use case begins: the same framework can model different inspection setups.

if illumination == "brightfield":
    source = bright_field(wavelength=532e-9, power=1.0, incidence_angle=0.0)
elif illumination == "darkfield":
    source = dark_field(wavelength=532e-9, power=2.0, incidence_angle=0.785)
else:
    raise ValueError(f"Unsupported illumination: {illumination}")

print(f"Source: {type(source).__name__}")
print(f"Propagation direction: {source.propagation_direction}")

In [ ]:
# Run the full simulation pipeline.
# The framework handles each stage and returns the intermediate outputs.
pipeline = SimulationPipeline(
    source=source,
    surface=surface,
    scattering=LambertianScattering(albedo=0.7),
    optics=OpticalSystem(focal_length=0.05, aperture_diameter=0.008, wavelength=532e-9),
    propagator=OpticalPropagator(GaussianPSF(sigma=1.0)),
    detector=CMOSDetector(exposure_time=exposure_time, gain=1.0),
    analysers=[DefectAnalyzer(threshold=threshold)],
)

result = pipeline.run(shape=shape, spacing=0.5, view_direction=np.array([0.0, 0.0, 1.0]))
print(result.describe())

In [ ]:
# Inspect the analysis result.
report = result.report
measurements = report.measurements if report is not None else {}
print("Analysis measurements:")
for name, value in measurements.items():
    if isinstance(value, float):
        print(f"  {name}: {value:.4f}")
    else:
        print(f"  {name}: {value}")

In [ ]:
# Visualise the detector output so the effect of the parameters is easy to see.
print(result.digital_image.visualize(max_width=64))

## Try next

Try changing one thing at a time:
- switch from darkfield to brightfield and compare the defect contrast
- increase or decrease the threshold to change the defect count
- change the defect type from dent to crack or stain
- increase the exposure time to see how the detector signal changes